## Step 9: Comparative Phylogenomics
**Input:** Whole-genome MAF alignment from Step 5 
(`06-scaffolding/cactus/alignment.maf`); 
proteomes of 8 Fusarium species for OrthoFinder  
**Output:** Neutral model (`neutral_model.mod`); conserved elements BED 
(`cons.bed`); phyloP scores (`phyloP_scores.wig`); 
OrthoFinder results in `11-comparative-genomics/orthofinder/results/`  
**Tools:** PHAST v1.9.7 (phyloFit, phastCons, phyloP), OrthoFinder v3.0.1  
**Key parameters:** phastCons: target-coverage 0.3, expected-length 45; 
phyloP: CONACC mode, LRT method; 
OrthoFinder: multiple sequence alignment-based orthogroup inference  
**Key finding:** 3,914 conserved elements (1.003 Mb); 96.3% of genes 
assigned to orthogroups; 106 species-specific orthogroups (245 genes); 
1,050 gene duplication events  
**Reference:** Materials & Methods Section 6 — Nebli et al. (2025)

# Set up

In [ ]:
# For PHAST tools
alias phyloFit="apptainer run docker://quay.io/comparative-genomics-toolkit/cactus:v2.9.9 phyloFit"
alias phyloP="apptainer run docker://quay.io/comparative-genomics-toolkit/cactus:v2.9.9 phyloP"
alias phastCons="apptainer run docker://quay.io/comparative-genomics-toolkit/cactus:v2.9.9 phastCons"

# For OrthoFinder
alias orthofinder="apptainer run docker://quay.io/biocontainers/orthofinder:3.1.0--hdfd78af_0 orthofinder"

# Generating a Neutral Model with phyloFit

In [ ]:
# We can use the tree from the scaffolding chapter
# The MAF file also comes from the scaffolding chapter
export TREE="(sample10-scaffolds,((LD-06,race4),(ME23,(ZUM2407,(V032g,(Fo47,fo5176))))));"
export MAF_FILE="06-scaffolding/cactus/alignment.maf"

echo "$TREE" > tree.nwk

# The tree needs to be in Newick format, e.g., "((sp1,sp2),sp3);"
# Let's assume we have a file `tree.nwk` with the tree.
#mkdir -p ~/tmp
export TMPDIR=~/tmp
phyloFit --tree tree.nwk --subst-mod REV --out-root neutral_model ${MAF_FILE}

# Predicting Conserved Elements with PHAST

In [ ]:
# Path to the MAF file from chapter 06
export MAF_FILE="06-scaffolding/cactus/alignment.maf"
export NEUTRAL_MODEL="neutral_model.mod"

phastCons --target-coverage 0.3 --expected-length 45 \
$MAF_FILE $NEUTRAL_MODEL \
--most-conserved cons.bed \
--score > cons.wig


# phyloP

In [ ]:
# Using the MAF file and the neutral model
export MAF_FILE="06-scaffolding/cactus/alignment.maf"
export NEUTRAL_MODEL="neutral_model.mod"

phyloP --mode CONACC --method LRT --wig-scores $NEUTRAL_MODEL $MAF_FILE > phyloP_scores.wig

# Set up for Orthofinder analysis

In [ ]:
# Create directories
mkdir -p 11-comparative-genomics/orthofinder/proteomes
mkdir -p 11-comparative-genomics/orthofinder/primary_transcripts

# Download proteomes
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/013/085/055/GCF_013085055.1_ASM1308505v1/GCF_013085055.1_ASM1308505v1_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/Fo47.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/030/719/095/GCA_030719095.1_ASM3071909v1/GCA_030719095.1_ASM3071909v1_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/ME23.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/240/135/GCF_000240135.3_ASM24013v3/GCF_000240135.3_ASM24013v3_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/F.graminearum.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/149/555/GCF_000149555.1_ASM14955v1/GCF_000149555.1_ASM14955v1_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/F.verticillioides.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/079/805/GCF_900079805.1_Fusarium_fujikuroi_IMI58289_V2/GCF_900079805.1_Fusarium_fujikuroi_IMI58289_V2_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/F.fujikuroi.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/025/331/925/GCA_025331925.1_ASM2533192v1/GCA_025331925.1_ASM2533192v1_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/Fo5176.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/149/955/GCF_000149955.1_ASM14995v2/GCF_000149955.1_ASM14995v2_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/Fol4287.faa.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/031/834/405/GCA_031834405.1_ASM3183440v1/GCA_031834405.1_ASM3183440v1_protein.faa.gz -O 11-comparative-genomics/orthofinder/proteomes/II5faa.gz

# Decompress files
gunzip 11-comparative-genomics/orthofinder/proteomes/*.gz

# Copy our annotated proteome 
cp 09-annot/prot_only/braker.aa 11-comparative-genomics/orthofinder/proteomes/our_strain.faa

# Download the script to select the longest transcript
wget https://raw.githubusercontent.com/davidemms/OrthoFinder/master/tools/primary_transcript.py -O 11-comparative-genomics/orthofinder/primary_transcript.py
chmod +x 11-comparative-genomics/orthofinder/primary_transcript.py

# Run the script on each proteome file
for f in 11-comparative-genomics/orthofinder/proteomes/*.faa; do
    python 11-comparative-genomics/orthofinder/primary_transcript.py $f
done

# Move the resulting primary transcript files to a dedicated directory
mv 11-comparative-genomics/orthofinder/proteomes/primary_transcripts/*.faa 11-comparative-genomics/orthofinder/primary_transcripts/

# Running OrthoFinder

In [ ]:
orthofinder -f 11-comparative-genomics/orthofinder/primary_transcripts/ -o 11-comparative-genomics/orthofinder/results/


orthofinder \
-f 11-comparative-genomics/orthofinder/primary_transcripts/ \
-o 11-comparative-genomics/orthofinder/results/